# ASO → IDT order format

Turn an antisense oligonucleotide — **sequence** + per-position **sugar chemistry** + **phosphorothioate (PS) pattern** — into an IDT-style order string, using the shared `tauso.common.modifications.to_idt_notation`.

- **Chemical (sugar) pattern** — one letter per base, so its length equals the sequence length: `M` = 2′-MOE · `C` = cEt · `L` = LNA · `d` = DNA
- **PS pattern** — one character per inter-nucleotide linkage, so its length equals *sequence length − 1*: `*` = phosphorothioate · anything else = phosphodiester. Omit it for an all-PS backbone.
- **5-methyl-cytosine** — pass `modification_string="MOE/5-methylcytosines/deoxy"` so every cytosine is 5-methyl: the DNA-gap C becomes `/iMe-dC/`, and the 2′-MOE wing C is already the 5-methyl form (`/i2MOErC/`).

`to_idt_notation` validates both lengths (chemical pattern = *L*, PS pattern = *L − 1*) and raises `ValueError` on a mismatch or an unsupported base/sugar.

> ⚠️ The emitted modification codes are a starting point — confirm them against IDT's current catalogue before ordering (cEt in particular is a placeholder token).

In [1]:
from tauso.common.modifications import to_idt_notation

def standard_gapmer(sequence, chemistry="MOE"):
    """Shortcut chemical pattern for a standard gapmer: modified wings + DNA gap, with the
    reasonable default wing size per chemistry (2'-MOE -> 5-nt wings, e.g. a 20-mer is 5-10-5;
    cEt -> 3-nt wings, e.g. a 16-mer is 3-10-3). `chemistry` defaults to 2'-MOE."""
    wing, code = {"MOE": (5, "M"), "cEt": (3, "C")}[chemistry]
    return code * wing + "d" * (len(sequence) - 2 * wing) + code * wing

## One ASO

Type a **sequence** and a **chemical pattern** (one sugar letter per base), then run. For a standard 2′-MOE or cEt gapmer you can skip typing the pattern and use `standard_gapmer(sequence)` / `standard_gapmer(sequence, "cEt")` instead. Pass `modification_string="MOE/5-methylcytosines/deoxy"` to 5-methylate every cytosine (the canonical gapmer convention).

In [2]:
sequence         = "GCAGTTATATTAGGTTCTCG"       # bases, 5'->3'
chemical_pattern = "MMMMMddddddddddMMMMM"        # one sugar per base: M=2'MOE  C=cEt  L=LNA  d=DNA
ps_pattern       = "*" * (len(sequence) - 1)     # optional; all-PS by default

print(to_idt_notation(sequence, chemical_pattern, ps_pattern))

# shortcut — a standard 2'-MOE gapmer gives the same pattern without typing it out:
print(to_idt_notation(sequence, standard_gapmer(sequence)))

# with 5-methyl-cytosine on every C (gap C -> /iMe-dC/, wing C already 5-methyl via /i2MOErC/):
print(to_idt_notation(sequence, standard_gapmer(sequence), modification_string="MOE/5-methylcytosines/deoxy"))

/52MOErG/*/i2MOErC/*/i2MOErA/*/i2MOErG/*/i2MOErT/*T*A*T*A*T*T*A*G*G*T*/i2MOErT/*/i2MOErC/*/i2MOErT/*/i2MOErC/*/32MOErG/
/52MOErG/*/i2MOErC/*/i2MOErA/*/i2MOErG/*/i2MOErT/*T*A*T*A*T*T*A*G*G*T*/i2MOErT/*/i2MOErC/*/i2MOErT/*/i2MOErC/*/32MOErG/
/52MOErG/*/i2MOErC/*/i2MOErA/*/i2MOErG/*/i2MOErT/*T*A*T*A*T*T*A*G*G*T*/i2MOErT/*/i2MOErC/*/i2MOErT/*/i2MOErC/*/32MOErG/


## MALAT1 top-3 — TAUSO and OligoAI

Top 3 predicted ASOs from each model for MALAT1 (A549 / Lipofection / 100 nM, 2′-MOE 5-10-5 gapmer, full PS, 5-methyl-cytosine on every C).

In [3]:
import pandas as pd

MOD = "MOE/5-methylcytosines/deoxy"   # every C is 5-methyl: wing C via /i2MOErC/, DNA-gap C via /iMe-dC/

# (model, rank, target_start, sequence 5'->3')
specs = [
    ("TAUSO",   1, 4637, "GCCACTTCCTTTGCTCTGCA"),
    ("TAUSO",   2, 5346, "TCTCATTTATTTCGGCTTCT"),
    ("TAUSO",   3, 5762, "CCTTAGTTGGCATCAAGGCA"),
    ("OligoAI", 1, 8664, "GCAGTTATATTAGGTTCTCG"),
    ("OligoAI", 2, 6984, "GGAATGTTTCTTGTCACTGA"),
    ("OligoAI", 3, 8729, "GTTAAGTTTTCAGCAGTAGG"),
]

pd.DataFrame(
    [{"model": m, "rank": r, "target_start": t, "sequence": s,
      "idt_order": to_idt_notation(s, standard_gapmer(s), modification_string=MOD)}
     for m, r, t, s in specs]
)

,model,rank,target_start,sequence,idt_order
0,TAUSO,1,4637,GCCACTTCCTTTGCTCTGCA,/52MOErG/*/i2MOErC/*/i2MOErC/*/i2MOErA/*/i2MOE...
1,TAUSO,2,5346,TCTCATTTATTTCGGCTTCT,/52MOErT/*/i2MOErC/*/i2MOErT/*/i2MOErC/*/i2MOE...
2,TAUSO,3,5762,CCTTAGTTGGCATCAAGGCA,/52MOErC/*/i2MOErC/*/i2MOErT/*/i2MOErT/*/i2MOE...
3,OligoAI,1,8664,GCAGTTATATTAGGTTCTCG,/52MOErG/*/i2MOErC/*/i2MOErA/*/i2MOErG/*/i2MOE...
4,OligoAI,2,6984,GGAATGTTTCTTGTCACTGA,/52MOErG/*/i2MOErG/*/i2MOErA/*/i2MOErA/*/i2MOE...
5,OligoAI,3,8729,GTTAAGTTTTCAGCAGTAGG,/52MOErG/*/i2MOErT/*/i2MOErT/*/i2MOErA/*/i2MOE...
